# 49 — Peer-message reflection-grounding smoke test

Verifies that `generate_peer_message()` and `generate_package_peer_message()` now
explicitly surface today's reflections in the user prompt (Option B change), so peer
messages are mechanistically grounded in what the agent heard from political broadcasts
rather than just drawing on reflections buried in the system-prompt context.

**No LLM server required for checks 1–7** — all structural assertions inspect the actual
user-prompt string captured from a mock `send_chat`. An optional §10 runs a real LLM call
if a local server is reachable.

**What this notebook checks**

1. No reflections today → no reflection block in user prompt.
2. One today-reflection → verbatim text appears in user prompt.
3. Multiple today-reflections → all appear, bulleted.
4. Stale reflections (earlier days) → excluded from prompt.
5. Cross-policy reflections → excluded from prompt.
6. Package mode, no reflections → no reflection block.
7. Package mode, today's reflections → verbatim text appears.
8. Scoreboard.
9. *(Optional)* Real LLM call — message text visibly references reflection content.

## 1. Imports + helpers

In [ ]:
import os, sys
from unittest.mock import patch, MagicMock

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '../src')))

from cag.abm.agent import SurveyedCitizen
from cag.abm.attributes.opinion import ClimatePolicyID, PACKAGE_SCOPE, SURVEY_QUESTIONS

_FAKE_MSG = "This is a simulated peer message response."
_CHECKS = {}  # label -> bool

def _make_citizen(reflections=None):
    """Minimal citizen with a mocked environment and optional seed reflections."""
    citizen = SurveyedCitizen(agent_id=0, environment=MagicMock())
    citizen.get_system_prompt = MagicMock(return_value="[system prompt]")
    citizen.reflections = list(reflections or [])
    return citizen

def _capture_user_prompt(citizen, method, *args, **kwargs):
    """Call `method` on `citizen`, patching send_chat, and return the user prompt."""
    with patch("cag.abm.agent.send_chat", return_value=_FAKE_MSG) as mock_send:
        method(*args, **kwargs)
        return mock_send.call_args[0][1]   # positional arg 1 = user_prompt

POLICY = ClimatePolicyID.CARBON_TAX
POLICY_DESC = SURVEY_QUESTIONS[POLICY]
print(f"Policy under test: {POLICY} — '{POLICY_DESC}'")
print("Helpers ready.")

## 2. Check 1 — no reflections today: no reflection block in prompt

In [ ]:
citizen = _make_citizen(reflections=[])

prompt = _capture_user_prompt(
    citizen, citizen.generate_peer_message,
    POLICY, day=1,
)

has_block = "Your recent reflections on this" in prompt
_CHECKS["no_reflections_no_block"] = not has_block

print(f"User prompt:\n{prompt}\n")
print(f"Reflection block absent: {not has_block}")
assert not has_block, "FAIL: reflection block should be absent when there are no reflections"
print("CHECK 1 PASSED")

## 3. Check 2 — one today-reflection: verbatim text appears in prompt

In [ ]:
REFL_TEXT = "The carbon tax seems quite radical to me, though I understand the rationale."

citizen = _make_citizen(reflections=[
    {"day": 1, "policy_id": POLICY, "text": REFL_TEXT},
])

prompt = _capture_user_prompt(
    citizen, citizen.generate_peer_message,
    POLICY, day=1,
)

has_block = "Your recent reflections on this" in prompt
has_text  = REFL_TEXT in prompt
_CHECKS["single_reflection_present"] = has_block and has_text

print(f"User prompt:\n{prompt}\n")
print(f"Reflection header present: {has_block}")
print(f"Reflection text present:   {has_text}")
assert has_block, "FAIL: reflection header missing"
assert has_text,  "FAIL: reflection text missing"
print("CHECK 2 PASSED")

## 4. Check 3 — multiple today-reflections: all appear, bulleted

In [ ]:
REFL_A = "The economic arguments in the broadcast gave me pause."
REFL_B = "I still think the environment needs protecting, but the costs worry me."

citizen = _make_citizen(reflections=[
    {"day": 2, "policy_id": POLICY, "text": REFL_A},
    {"day": 2, "policy_id": POLICY, "text": REFL_B},
])

prompt = _capture_user_prompt(
    citizen, citizen.generate_peer_message,
    POLICY, day=2,
)

has_a = REFL_A in prompt
has_b = REFL_B in prompt
bullet_count = prompt.count("\n- ")
_CHECKS["multiple_reflections_all_present"] = has_a and has_b

print(f"User prompt:\n{prompt}\n")
print(f"Reflection A present: {has_a}")
print(f"Reflection B present: {has_b}")
print(f"Bullet lines found:   {bullet_count}")
assert has_a, "FAIL: first reflection text missing"
assert has_b, "FAIL: second reflection text missing"
assert bullet_count >= 2, "FAIL: expected at least 2 bulleted lines"
print("CHECK 3 PASSED")

## 5. Check 4 — stale reflections (earlier days) are excluded

In [ ]:
OLD_REFL   = "Yesterday I thought the policy was too expensive."
TODAY_REFL = "Today's broadcast shifted my view slightly."

citizen = _make_citizen(reflections=[
    {"day": 1, "policy_id": POLICY, "text": OLD_REFL},   # yesterday
    {"day": 2, "policy_id": POLICY, "text": TODAY_REFL}, # today
])

prompt = _capture_user_prompt(
    citizen, citizen.generate_peer_message,
    POLICY, day=2,  # "today" is day 2
)

old_excluded   = OLD_REFL not in prompt
today_included = TODAY_REFL in prompt
_CHECKS["stale_reflections_excluded"] = old_excluded and today_included

print(f"User prompt:\n{prompt}\n")
print(f"Stale reflection excluded: {old_excluded}")
print(f"Today reflection included: {today_included}")
assert old_excluded,   "FAIL: reflection from previous day should not appear"
assert today_included, "FAIL: today's reflection should appear"
print("CHECK 4 PASSED")

## 6. Check 5 — cross-policy reflections are excluded

In [ ]:
OTHER_POLICY = ClimatePolicyID.RENEWABLE_ENERGY
CROSS_REFL   = "The renewable energy broadcast was very compelling."
CORRECT_REFL = "The carbon tax broadcast made me reconsider."

citizen = _make_citizen(reflections=[
    {"day": 1, "policy_id": OTHER_POLICY, "text": CROSS_REFL},   # wrong policy
    {"day": 1, "policy_id": POLICY,       "text": CORRECT_REFL},
])

prompt = _capture_user_prompt(
    citizen, citizen.generate_peer_message,
    POLICY, day=1,
)

cross_excluded   = CROSS_REFL not in prompt
correct_included = CORRECT_REFL in prompt
_CHECKS["cross_policy_reflections_excluded"] = cross_excluded and correct_included

print(f"User prompt:\n{prompt}\n")
print(f"Cross-policy reflection excluded:  {cross_excluded}")
print(f"Correct-policy reflection present: {correct_included}")
assert cross_excluded,   "FAIL: reflection for a different policy should not appear"
assert correct_included, "FAIL: same-policy reflection should appear"
print("CHECK 5 PASSED")

## 7. Check 6 — package mode, no reflections: no reflection block

In [ ]:
POLICY_IDS = [ClimatePolicyID.CARBON_TAX, ClimatePolicyID.RENEWABLE_ENERGY]

citizen = _make_citizen(reflections=[])

prompt = _capture_user_prompt(
    citizen, citizen.generate_package_peer_message,
    POLICY_IDS, day=1,
)

has_block = "Your recent reflections on this package" in prompt
_CHECKS["package_no_reflections_no_block"] = not has_block

print(f"User prompt:\n{prompt}\n")
print(f"Reflection block absent: {not has_block}")
assert not has_block, "FAIL: reflection block should be absent in package mode with no reflections"
print("CHECK 6 PASSED")

## 8. Check 7 — package mode, today's reflections appear

In [ ]:
PKG_REFL = "The package broadcast highlighted the housing insulation angle — that resonated."

citizen = _make_citizen(reflections=[
    {"day": 1, "policy_id": PACKAGE_SCOPE, "text": PKG_REFL},
])

prompt = _capture_user_prompt(
    citizen, citizen.generate_package_peer_message,
    POLICY_IDS, day=1,
)

has_header = "Your recent reflections on this package" in prompt
has_text   = PKG_REFL in prompt
_CHECKS["package_reflection_present"] = has_header and has_text

print(f"User prompt:\n{prompt}\n")
print(f"Package reflection header present: {has_header}")
print(f"Package reflection text present:   {has_text}")
assert has_header, "FAIL: package reflection header missing"
assert has_text,   "FAIL: package reflection text missing"
print("CHECK 7 PASSED")

## 9. Scoreboard

In [ ]:
print("=" * 62)
print("SMOKE TEST SCOREBOARD — peer-message reflection grounding")
print("=" * 62)
all_passed = True
for label, result in _CHECKS.items():
    status = "PASS" if result else "FAIL"
    if not result:
        all_passed = False
    print(f"  [{status}]  {label}")
print("=" * 62)
if all_passed:
    print(f"ALL {len(_CHECKS)} CHECKS PASSED")
else:
    failed = sum(1 for v in _CHECKS.values() if not v)
    print(f"{failed}/{len(_CHECKS)} CHECKS FAILED — see cells above for details.")

## 10. (Optional) Real LLM call — message text echoes reflection content

Requires a local LLM server on `http://localhost:8080/v1`. Skipped automatically if unreachable.

Seeds a citizen with a vivid, distinctive reflection and checks that the generated peer
message visibly echoes the core framing from that reflection — confirming the LLM actually
draws on the surfaced reflection rather than generating generic policy opinion.

In [ ]:
import requests as _req

SERVER = "http://localhost:8080/v1"
try:
    _req.get(SERVER + "/models", timeout=3)
    server_up = True
except Exception:
    server_up = False

if not server_up:
    print(f"Local LLM server not reachable at {SERVER} — skipping optional check.")
else:
    print(f"Server reachable. Running real LLM peer-message generation...")

    # Build a minimal citizen with a real persona (no YouGov load needed)
    llm_citizen = _make_citizen(reflections=[
        {
            "day": 1,
            "policy_id": POLICY,
            "text": (
                "The broadcast made a striking point: a carbon tax puts the burden "
                "squarely on ordinary households paying energy bills, not on corporations. "
                "That seems deeply unfair to me."
            ),
        }
    ])

    # Capture the user prompt to verify structure, then run the real call
    with patch("cag.abm.agent.send_chat", return_value=_FAKE_MSG) as mock_send:
        llm_citizen.generate_peer_message(POLICY, day=1)
        captured_prompt = mock_send.call_args[0][1]

    print(f"\n--- User prompt that will be sent ---\n{captured_prompt}\n")

    # Now run the actual LLM call (no patch)
    llm_citizen2 = _make_citizen(reflections=[
        {
            "day": 1,
            "policy_id": POLICY,
            "text": (
                "The broadcast made a striking point: a carbon tax puts the burden "
                "squarely on ordinary households paying energy bills, not on corporations. "
                "That seems deeply unfair to me."
            ),
        }
    ])
    msg = llm_citizen2.generate_peer_message(
        POLICY, day=1,
        provider="local",
        model="mlx-community/Qwen3-8B-4bit",
    )
    print(f"--- Generated peer message ---\n{msg}\n")

    # Soft check: message should reference the reflection's framing
    keywords = ["burden", "household", "fair", "cost", "tax", "ordinary", "corporation"]
    hits = [kw for kw in keywords if kw.lower() in msg.lower()]
    print(f"Keyword hits from reflection framing: {hits}")
    if hits:
        print("Optional check PASSED — message echoes reflection content.")
    else:
        print("Optional check inconclusive — no explicit keyword match; review message manually.")